In [ ]:
%pip install langchain langchain-community langchain-ollama langchain-text-splitters chromadb pypdf beautifulsoup4 lxml -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import re
from pathlib import Path
from bs4 import BeautifulSoup  


def extract_paragraphs_from_html(html_path: str) -> list[str]:
    """Extracts text from <p> tags"""
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")

    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(separator=" ", strip=True)
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            paragraphs.append(text)

    return paragraphs


def chunk_paragraphs(paragraphs: list[str], max_words: int = 400) -> list[str]:
    """Groups paragraphs"""

    chunks = []
    current_chunk = []
    current_word_count = 0

    for para in paragraphs:
        word_count = len(para.split())

        # skip junk
        if word_count < 5:
            continue

        if current_word_count + word_count > max_words and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            current_chunk = []
            current_word_count = 0

        current_chunk.append(para)
        current_word_count += word_count

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


def save_chunks(chunks: list[str], out_path: str):
    Path(out_path).write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved {len(chunks)} chunk(s) to {out_path}")


if __name__ == "__main__":
    HTML_PATH = "riscv-spec.html"  
    OUT_PATH = "chunks.json"
    MAX_WORDS = 400


    LIMIT = None

    paragraphs = extract_paragraphs_from_html(HTML_PATH)
    chunks = chunk_paragraphs(paragraphs, max_words=MAX_WORDS)

    print(f"Extracted {len(paragraphs)} paragraphs \n {len(chunks)} total chunks")

    if LIMIT is not None:
        chunks = chunks[:LIMIT]
        print(f"LIMIT set: only saving first {len(chunks)} chunk(s) for review")

    save_chunks(chunks, OUT_PATH)


Extracted 18357 paragraphs -> 577 total chunks
Saved 577 chunk(s) to chunks.json


In [ ]:
"Run model on extracted chunks and produce JSON of targeted "

import json
import sys

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# ollama pull granite3.3:8b
llm = ChatOllama(model="granite3.3:8b", temperature=0)

template = """You are given a snippet of text from an Instruction Set Architecture (ISA) manual.

Your task: identify every sentence in the snippet that describes optional, discretionary,
implementation-defined, or vendor/platform-dependent behavior as opposed to mandatory
requirements (e.g. "shall", "must", "is required to").

Look for signals such as:
- Permission language ("may", "can", "is permitted to", "at the discretion of")
- Optionality ("optional", "if supported", "when available")
- Implementation deferral ("implementation-defined", "implementation-dependent",
  "vendor-specific", "platform-specific", "architecture-specific")
- Conventions or recommendations rather than requirements ("by convention",
  "it is recommended", "typically", "commonly")
- Variability across systems ("can vary", "may differ", "is not guaranteed to")

Do NOT flag sentences that use mandatory language ("shall", "must", "is required",
"will always"), even if they appear near flagged sentences.

Do NOT flag sentences that are purely descriptive of a fixed encoding, structure, or
behavior, even if they mention bit ranges, fields, or values unless the sentence itself
states that the value, size, or behavior is left to the implementation, convention, or
vendor to decide.


Examples:

Sentence: "Cache block sizes are implementation dependent and can vary across platforms."
Flag: YES
Reason: Explicitly defers the cache block size to the implementation and states it can vary.
Category: implementation_deferral

Sentence: "The top two bits (csr[11:10]) indicate whether the register is read/write
(00, 01, or 10) or read-only (11)."
Flag: NO
Reason: This describes a fixed, already-defined encoding scheme. It states what the bits
mean, not that the encoding itself is optional, discretionary, or left to the implementation.


For each sentence you flag, respond with a JSON object containing:
- "sentence": the exact sentence text, unmodified
- "reason": a brief explanation (1 sentence) of why it signals optional/deferred behavior
- "category": one of ["permission_modal", "optionality", "implementation_deferral",
  "convention_or_recommendation", "variability"]

Respond ONLY with a JSON array of these objects. If no sentences qualify, respond with
an empty array: []. Do not include any preamble, explanation, or markdown formatting
outside the JSON array.

--- BEGIN SNIPPET ---
{snippet}
--- END SNIPPET ---
"""

prompt = ChatPromptTemplate.from_template(template)
extraction_chain = prompt | llm | JsonOutputParser()


def extract(snippet: str, debug: bool = True) -> dict:
    if debug:
        raw = (prompt | llm).invoke({"snippet": snippet})
        print("----- RAW -----", file=sys.stderr)
        print(raw.content, file=sys.stderr)
        print("---------------", file=sys.stderr)

    try:
        return extraction_chain.invoke({"snippet": snippet})
    except Exception as e:
        sys.exit(
            f"Error: {e}\n"
            "Make sure Ollama is running and 'granite3.3:8b' has been pulled "
        )


if __name__ == "__main__":

    #snippet1 = """Privileged Spec 19.3.1: 
#Caches organize copies of data into cache blocks, each of which represents a contiguous, naturally aligned power-of-two (or NAPOT) range of memory locations. A cache block is identified by any of the physical addresses corresponding to the underlying memory locations. The capacity and organization of a cache and the size of a cache block are both implementation-specific, and the execution environment provides software a means to discover information about the caches and cache blocks in a system. In the initial set of CMO extensions, the size of a cache block shall be uniform throughout the system.
#"""
 #   snippet2 = """Privileged Spec 2.1: 
#"Conventional" R/W accessibility of CSRs according to address mapping 
#The standard RISC-V ISA sets aside a 12-bit encoding space (csr[11:0]) for up to 4,096 CSRs. By convention, the upper 4 bits of the CSR address (csr[11:8]) are used to encode the read and write accessibility of the CSRs according to privilege level as shown in Table 1. The top two bits (csr[11:10]) indicate whether the register is read/write (00,01, or 10) or read-only (11). The next two bits (csr[9:8]) encode the lowest privilege level that can access the CSR.
#"""

    import json

    with open("chunks.json", encoding="utf-8") as f:
        data = json.load(f)

    all_results = []
    LLM_LIMIT = 10
    for i, chunk in enumerate(data, 1):
        print(f"[{i}/{len(data)}] Processing chunk...")
        result = extract(chunk, debug=False)  # returns a list of match dicts
        all_results.extend(result)
        LLM_LIMIT = LLM_LIMIT - 1
        if LLM_LIMIT == 0:
            break

    print(f"\nFound {len(all_results)} flagged sentences total\n")
    print(json.dumps(all_results, indent=2))

    with open("freedom_matches.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print("Saved to freedom_matches.json")





[1/577] Processing chunk...
[2/577] Processing chunk...
[3/577] Processing chunk...
[4/577] Processing chunk...
[5/577] Processing chunk...
[6/577] Processing chunk...
[7/577] Processing chunk...
[8/577] Processing chunk...
[9/577] Processing chunk...
[10/577] Processing chunk...

Found 48 flagged sentences total

[
  {
    "sentence": "Addition of the Zalasr extension for Load-Acquire/Store-Release operations.",
    "reason": "This sentence describes an addition, implying it's optional or discretionary as not all RISC-V implementations may support or choose to implement this feature.",
    "category": "optionality"
  },
  {
    "sentence": "The inclusion of all ratified extensions through May 2025.",
    "reason": "This sentence indicates the inclusion is conditional on extensions being ratified, suggesting optionality based on ratification status.",
    "category": "implementation_deferral"
  },
  {
    "sentence": "Addition of the BFloat16-precision Floating Point extension.",
    "

: 

In [ ]:
"""
Converts freedom_matches.json into YAML file
"""

import json
import re
import yaml


def make_slug(sentence: str, max_words: int = 6) -> str:
    words = re.findall(r"[A-Za-z0-9]+", sentence.lower())
    slug = "-".join(words[:max_words])
    return slug or "unnamed-entry"


def dedupe_slug(slug: str, seen: dict) -> str:
    if slug not in seen:
        seen[slug] = 1
        return slug
    seen[slug] += 1
    return f"{slug}-{seen[slug]}"


def convert(matches: list[dict]) -> list[dict]:
    seen_slugs = {}
    entries = []

    for m in matches:
        sentence = m.get("sentence", "").strip()
        reason = m.get("reason", "").strip()
        category = m.get("category", "uncategorized")

        slug = dedupe_slug(make_slug(sentence), seen_slugs)

        entries.append({
            "name": slug,
            "description": sentence,
            "type": category,
            "constraints": reason,
        })

    return entries


if __name__ == "__main__":
    IN_PATH = "freedom_matches.json"
    OUT_PATH = "freedom_matches.yaml"

    with open(IN_PATH, encoding="utf-8") as f:
        matches = json.load(f)

    entries = convert(matches)

    with open(OUT_PATH, "w", encoding="utf-8") as f:
        yaml.dump(
            entries,
            f,
            sort_keys=False,
            allow_unicode=True,
            default_flow_style=False,
            width=100,
        )

    print(f"Converted {len(entries)} entries to {OUT_PATH}")

Converted 48 entries -> freedom_matches.yaml
